# Week 3 — Data Integration & Feature Engineering

## Objective

The purpose of this notebook is to integrate the cleaned CodeTX datasets into analysis-ready tables and create useful features for later business analysis and predictive modeling.

## Week 3 Goals

- Load cleaned datasets
- Validate join keys before merging
- Create integrated sales and procurement datasets
- Create inventory and warehouse analytical tables
- Engineer useful business features
- Avoid double-counting during joins
- Prepare reusable datasets for later analysis
- Document all integration decisions

In [1]:
import pandas as pd
import numpy as np

In [2]:
branches = pd.read_csv("processed_data/branches_clean.csv")
customers = pd.read_csv("processed_data/customers_clean.csv")
inventory = pd.read_csv("processed_data/inventory_clean.csv")
invoices = pd.read_csv("processed_data/invoices_clean.csv")
payments = pd.read_csv("processed_data/payments_clean.csv")
products = pd.read_csv("processed_data/products_clean.csv")
po_header = pd.read_csv("processed_data/po_header_clean.csv")
po_lines = pd.read_csv("processed_data/po_lines_clean.csv")
so_header = pd.read_csv("processed_data/so_header_clean.csv")
so_lines = pd.read_csv("processed_data/so_lines_clean.csv")
stock_ledger = pd.read_csv("processed_data/stock_ledger_clean.csv")
suppliers = pd.read_csv("processed_data/suppliers_clean.csv")

In [3]:
datasets = {
    "branches": branches,
    "customers": customers,
    "inventory": inventory,
    "invoices": invoices,
    "payments": payments,
    "products": products,
    "po_header": po_header,
    "po_lines": po_lines,
    "so_header": so_header,
    "so_lines": so_lines,
    "stock_ledger": stock_ledger,
    "suppliers": suppliers
}

In [4]:
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

branches: (6, 14)
customers: (500, 15)
inventory: (180, 8)
invoices: (18033, 12)
payments: (19257, 8)
products: (30, 23)
po_header: (24000, 10)
po_lines: (155495, 9)
so_header: (20000, 11)
so_lines: (130402, 9)
stock_ledger: (237230, 9)
suppliers: (8, 12)


In [5]:
date_columns = {
    "customers": ["customer_since", "last_purchase_date"],
    "invoices": ["invoice_date", "due_date"],
    "payments": ["payment_date"],
    "products": ["last_purchase_date"],
    "po_header": ["order_date", "expected_delivery_date", "received_date"],
    "so_header": ["order_date", "delivery_date"],
    "stock_ledger": ["movement_date"]
}

for dataset_name, columns in date_columns.items():
    for column in columns:
        datasets[dataset_name][column] = pd.to_datetime(
            datasets[dataset_name][column],
            errors="coerce"
        )

In [6]:
sales_analysis = so_lines.copy()

In [7]:
sales_analysis = sales_analysis.merge(
    so_header[
        [
            "so_id",
            "customer_id",
            "branch_id",
            "order_date",
            "delivery_date",
            "order_status",
            "payment_terms",
            "sales_channel"
        ]
    ],
    on="so_id",
    how="left",
    validate="many_to_one"
)

In [8]:
sales_analysis = sales_analysis.merge(
    products[
        [
            "product_id",
            "product_name",
            "category",
            "machine_type",
            "brand",
            "unit_cost",
            "unit_price",
            "criticality_level",
            "usage_frequency",
            "lead_time_days"
        ]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)

In [9]:
sales_analysis = sales_analysis.merge(
    customers[
        [
            "customer_id",
            "customer_type",
            "industry_segment",
            "region",
            "customer_rating"
        ]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

In [10]:
sales_analysis = sales_analysis.rename(
    columns={"region": "customer_region"}
)

In [11]:
sales_analysis = sales_analysis.merge(
    branches[
        [
            "branch_id",
            "branch_name",
            "city",
            "region",
            "warehouse_type",
            "market_demand_index"
        ]
    ],
    on="branch_id",
    how="left",
    validate="many_to_one"
)

In [12]:
sales_analysis = sales_analysis.rename(
    columns={"region": "branch_region"}
)

In [13]:
print("Original sales line rows:", len(so_lines))
print("Integrated sales rows:", len(sales_analysis))
print("Difference:", len(sales_analysis) - len(so_lines))

Original sales line rows: 130402
Integrated sales rows: 130402
Difference: 0


In [14]:
sales_analysis.shape

(130402, 34)

In [15]:
sales_analysis.head()

,so_id,line_number,product_id,quantity,unit_price_x,gst_rate,line_total,gst_amount,line_grand_total,customer_id,...,lead_time_days,customer_type,industry_segment,customer_region,customer_rating,branch_name,city,branch_region,warehouse_type,market_demand_index
0,SO-990591,1,P018,19,11300,28,214700,60116.0,274816.0,C0070,...,20,Government,Logistics,West,3,Ahmedabad West Hub,Ahmedabad,West,Service Hub,7
1,SO-990591,2,P026,2,310,18,620,111.6,731.6,C0070,...,4,Government,Logistics,West,3,Ahmedabad West Hub,Ahmedabad,West,Service Hub,7
2,SO-990591,3,P026,3,310,18,930,167.4,1097.4,C0070,...,4,Government,Logistics,West,3,Ahmedabad West Hub,Ahmedabad,West,Service Hub,7
3,SO-990591,4,P013,9,1450,18,13050,2349.0,15399.0,C0070,...,9,Government,Logistics,West,3,Ahmedabad West Hub,Ahmedabad,West,Service Hub,7
4,SO-990591,5,P004,3,39800,28,119400,33432.0,152832.0,C0070,...,30,Government,Logistics,West,3,Ahmedabad West Hub,Ahmedabad,West,Service Hub,7


In [16]:
sales_analysis["sales_year"] = sales_analysis["order_date"].dt.year
sales_analysis["sales_month"] = sales_analysis["order_date"].dt.month
sales_analysis["sales_month_name"] = sales_analysis["order_date"].dt.month_name()
sales_analysis["sales_quarter"] = sales_analysis["order_date"].dt.quarter

In [17]:
sales_analysis["estimated_cost"] = (
    sales_analysis["quantity"] * sales_analysis["unit_cost"]
)

sales_analysis["gross_profit"] = (
    sales_analysis["line_total"] - sales_analysis["estimated_cost"]
)

sales_analysis["gross_margin_percent"] = (
    sales_analysis["gross_profit"]
    / sales_analysis["line_total"]
    * 100
)

In [18]:
sales_analysis[
    [
        "so_id",
        "product_id",
        "quantity",
        "line_total",
        "estimated_cost",
        "gross_profit",
        "gross_margin_percent",
        "sales_year",
        "sales_month_name",
        "sales_quarter"
    ]
].head(10)

,so_id,product_id,quantity,line_total,estimated_cost,gross_profit,gross_margin_percent,sales_year,sales_month_name,sales_quarter
0,SO-990591,P018,19,214700,155800,58900,27.433628,2022,May,2
1,SO-990591,P026,2,620,360,260,41.935484,2022,May,2
2,SO-990591,P026,3,930,540,390,41.935484,2022,May,2
3,SO-990591,P013,9,13050,8550,4500,34.482759,2022,May,2
4,SO-990591,P004,3,119400,97500,21900,18.341709,2022,May,2
5,SO-990591,P001,9,220500,170550,49950,22.653061,2022,May,2
6,SO-990591,P021,3,138600,106800,31800,22.943723,2022,May,2
7,SO-990591,P010,11,383900,317900,66000,17.191977,2022,May,2
8,SO-990591,P001,10,245000,189500,55500,22.653061,2022,May,2
9,SO-990591,P026,16,4960,2880,2080,41.935484,2022,May,2


In [19]:
print(
    "Negative gross profit rows:",
    (sales_analysis["gross_profit"] < 0).sum()
)

print(
    "Zero revenue rows:",
    (sales_analysis["line_total"] == 0).sum()
)

print(
    "Missing margin values:",
    sales_analysis["gross_margin_percent"].isna().sum()
)

Negative gross profit rows: 0
Zero revenue rows: 0
Missing margin values: 0


In [20]:
procurement_analysis = po_lines.copy()

In [21]:
procurement_analysis = procurement_analysis.merge(
    po_header[
        [
            "po_id",
            "supplier_id",
            "branch_id",
            "order_date",
            "expected_delivery_date",
            "received_date",
            "po_status"
        ]
    ],
    on="po_id",
    how="left",
    validate="many_to_one"
)

In [22]:
procurement_analysis = procurement_analysis.merge(
    products[
        [
            "product_id",
            "product_name",
            "category",
            "machine_type",
            "brand",
            "criticality_level",
            "usage_frequency",
            "lead_time_days"
        ]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)

In [23]:
procurement_analysis = procurement_analysis.rename(
    columns={"lead_time_days": "product_lead_time_days"}
)

In [24]:
procurement_analysis = procurement_analysis.merge(
    suppliers[
        [
            "supplier_id",
            "supplier_name",
            "supplier_type",
            "product_category",
            "region",
            "lead_time_days",
            "reliability_score",
            "import_duty_rate"
        ]
    ],
    on="supplier_id",
    how="left",
    validate="many_to_one"
)

In [25]:
procurement_analysis = procurement_analysis.rename(
    columns={
        "region": "supplier_region",
        "lead_time_days": "supplier_lead_time_days"
    }
)

In [26]:
procurement_analysis = procurement_analysis.merge(
    branches[
        [
            "branch_id",
            "branch_name",
            "city",
            "region",
            "warehouse_type",
            "warehouse_capacity_sqft"
        ]
    ],
    on="branch_id",
    how="left",
    validate="many_to_one"
)

In [27]:
procurement_analysis = procurement_analysis.rename(
    columns={"region": "branch_region"}
)

In [28]:
print("Original PO line rows:", len(po_lines))
print("Integrated procurement rows:", len(procurement_analysis))
print("Difference:", len(procurement_analysis) - len(po_lines))

Original PO line rows: 155495
Integrated procurement rows: 155495
Difference: 0


In [29]:
procurement_analysis.shape

(155495, 34)

In [30]:
procurement_analysis.head()

,po_id,line_number,product_id,quantity,unit_cost,gst_rate,line_total,gst_amount,line_grand_total,supplier_id,...,product_category,supplier_region,supplier_lead_time_days,reliability_score,import_duty_rate,branch_name,city,branch_region,warehouse_type,warehouse_capacity_sqft
0,PO-141186,1,P004,234,34709.54,28,8122032.36,2.274169e+06,1.039620e+07,SUP0001,...,Medium Parts,South China,19,5,8,Pune Distribution,Pune,West,Regional,38750
1,PO-141186,2,P022,79,12317.84,28,973109.36,2.724706e+05,1.245580e+06,SUP0001,...,Medium Parts,South China,19,5,8,Pune Distribution,Pune,West,Regional,38750
2,PO-141186,3,P030,46,13840.45,28,636660.70,1.782650e+05,8.149257e+05,SUP0001,...,Medium Parts,South China,19,5,8,Pune Distribution,Pune,West,Regional,38750
3,PO-141186,4,P001,293,16988.41,28,4977604.13,1.393729e+06,6.371333e+06,SUP0001,...,Medium Parts,South China,19,5,8,Pune Distribution,Pune,West,Regional,38750
4,PO-141186,5,P014,126,65443.06,28,8245825.56,2.308831e+06,1.055466e+07,SUP0001,...,Medium Parts,South China,19,5,8,Pune Distribution,Pune,West,Regional,38750


In [31]:
procurement_analysis["delivery_delay_days"] = (
    procurement_analysis["received_date"]
    - procurement_analysis["expected_delivery_date"]
).dt.days

In [32]:
procurement_analysis["actual_lead_time_days"] = (
    procurement_analysis["received_date"]
    - procurement_analysis["order_date"]
).dt.days

In [33]:
procurement_analysis["purchase_year"] = (
    procurement_analysis["order_date"].dt.year
)

procurement_analysis["purchase_month"] = (
    procurement_analysis["order_date"].dt.month
)

procurement_analysis["purchase_quarter"] = (
    procurement_analysis["order_date"].dt.quarter
)

In [34]:
procurement_analysis["delivery_performance"] = np.select(
    [
        procurement_analysis["po_status"].eq("Cancelled"),
        procurement_analysis["delivery_delay_days"] < 0,
        procurement_analysis["delivery_delay_days"] == 0,
        procurement_analysis["delivery_delay_days"] > 0
    ],
    [
        "Cancelled",
        "Early",
        "On Time",
        "Late"
    ],
    default="Unknown"
)

In [35]:
procurement_analysis["delivery_performance"].value_counts()

delivery_performance
On Time      112736
Late          21082
Cancelled     15418
Early          6259
Name: count, dtype: int64

In [36]:
po_header["delivery_delay_days"] = (
    po_header["received_date"]
    - po_header["expected_delivery_date"]
).dt.days

po_header["delivery_performance"] = np.select(
    [
        po_header["po_status"].eq("Cancelled"),
        po_header["delivery_delay_days"] < 0,
        po_header["delivery_delay_days"] == 0,
        po_header["delivery_delay_days"] > 0
    ],
    [
        "Cancelled",
        "Early",
        "On Time",
        "Late"
    ],
    default="Unknown"
)

po_header["delivery_performance"].value_counts()

delivery_performance
On Time      17392
Late          3267
Cancelled     2370
Early          971
Name: count, dtype: int64

In [37]:
po_delivery_summary = (
    po_header["delivery_performance"]
    .value_counts()
    .to_frame("po_count")
)

po_delivery_summary["percentage"] = (
    po_delivery_summary["po_count"]
    / len(po_header)
    * 100
).round(2)

po_delivery_summary

,po_count,percentage
delivery_performance,,
On Time,17392,72.47
Late,3267,13.61
Cancelled,2370,9.88
Early,971,4.05


### Procurement Dataset Grain

The integrated procurement dataset contains one row per purchase-order line rather than one row per purchase order.

Therefore, delivery-performance counts in `procurement_analysis` represent line items and should not be interpreted directly as counts of purchase orders.

Purchase-order-level delivery KPIs will be calculated using `po_header`, while the integrated line-level dataset will be used for product, supplier, category, and procurement-cost analysis.

Maintaining the correct analytical grain prevents double counting and incorrect business conclusions.

### Purchase Order Delivery Performance

Delivery performance was calculated at the purchase-order level to avoid double counting caused by multiple line items within each order.

The results show:

- 72.47% of purchase orders were delivered on time.
- 13.61% were delivered late.
- 4.05% were delivered earlier than expected.
- 9.88% were cancelled.

These results provide an initial view of procurement performance. Supplier-level delivery reliability will be investigated in a later analysis phase.

In [38]:
inventory_analysis = inventory.copy()

In [39]:
inventory_analysis = inventory_analysis.merge(
    products[
        [
            "product_id",
            "product_name",
            "category",
            "machine_type",
            "brand",
            "unit_cost",
            "unit_price",
            "criticality_level",
            "usage_frequency",
            "lead_time_days"
        ]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)

In [40]:
inventory_analysis = inventory_analysis.merge(
    branches[
        [
            "branch_id",
            "branch_name",
            "city",
            "region",
            "warehouse_type",
            "warehouse_capacity_sqft",
            "total_employees",
            "avg_monthly_revenue",
            "monthly_operational_cost",
            "market_demand_index"
        ]
    ],
    on="branch_id",
    how="left",
    validate="many_to_one"
)

In [41]:
inventory_analysis = inventory_analysis.rename(
    columns={"region": "branch_region"}
)

In [42]:
print("Original inventory rows:", len(inventory))
print("Integrated inventory rows:", len(inventory_analysis))
print("Difference:", len(inventory_analysis) - len(inventory))

print("Shape:", inventory_analysis.shape)

Original inventory rows: 180
Integrated inventory rows: 180
Difference: 0
Shape: (180, 26)


In [43]:
inventory_analysis["excess_stock_units"] = np.maximum(
    inventory_analysis["current_stock"] - inventory_analysis["max_stock"],
    0
)

In [44]:
inventory_analysis["stock_above_reorder"] = (
    inventory_analysis["current_stock"]
    - inventory_analysis["reorder_level"]
)

In [45]:
inventory_analysis["inventory_value"] = (
    inventory_analysis["current_stock"]
    * inventory_analysis["unit_cost"]
)

In [46]:
inventory_analysis["stock_status"] = np.select(
    [
        inventory_analysis["current_stock"]
        <= inventory_analysis["safety_stock"],

        inventory_analysis["current_stock"]
        <= inventory_analysis["reorder_level"],

        inventory_analysis["current_stock"]
        <= inventory_analysis["max_stock"]
    ],
    [
        "Critical",
        "Reorder Required",
        "Healthy"
    ],
    default="Overstock"
)

In [47]:
inventory_analysis["stock_status"].value_counts()

stock_status
Overstock    180
Name: count, dtype: int64

In [48]:
inventory_analysis["stock_to_max_ratio"] = (
    inventory_analysis["current_stock"]
    / inventory_analysis["max_stock"]
)

In [49]:
inventory_analysis["excess_inventory_value"] = (
    inventory_analysis["excess_stock_units"]
    * inventory_analysis["unit_cost"]
)

In [50]:
print(inventory_analysis.columns.tolist())

['product_id', 'branch_id', 'opening_stock', 'reorder_level', 'safety_stock', 'max_stock', 'current_stock', 'warehouse_bin', 'product_name', 'category', 'machine_type', 'brand', 'unit_cost', 'unit_price', 'criticality_level', 'usage_frequency', 'lead_time_days', 'branch_name', 'city', 'branch_region', 'warehouse_type', 'warehouse_capacity_sqft', 'total_employees', 'avg_monthly_revenue', 'monthly_operational_cost', 'market_demand_index', 'excess_stock_units', 'stock_above_reorder', 'inventory_value', 'stock_status', 'stock_to_max_ratio', 'excess_inventory_value']


In [51]:
inventory_analysis[
    [
        "product_id",
        "branch_id",
        "current_stock",
        "max_stock",
        "excess_stock_units",
        "inventory_value",
        "excess_inventory_value",
        "stock_to_max_ratio",
        "stock_status"
    ]
].head(10)

,product_id,branch_id,current_stock,max_stock,excess_stock_units,inventory_value,excess_inventory_value,stock_to_max_ratio,stock_status
0,P001,DEL001,115928,340,115588,2196835600,2190392600,340.964706,Overstock
1,P001,PUN001,109389,349,109040,2072921550,2066308000,313.435530,Overstock
2,P001,CHN001,99373,311,99062,1883118350,1877224900,319.527331,Overstock
3,P001,HYD001,108772,417,108355,2061229400,2053327250,260.844125,Overstock
4,P001,KOL001,103808,475,103333,1967161600,1958160350,218.543158,Overstock
5,P001,AHM001,106714,455,106259,2022230300,2013608050,234.536264,Overstock
6,P002,DEL001,108628,261,108367,48882600,48765150,416.199234,Overstock
7,P002,PUN001,101623,180,101443,45730350,45649350,564.572222,Overstock
8,P002,CHN001,105885,465,105420,47648250,47439000,227.709677,Overstock
9,P002,HYD001,121013,383,120630,54455850,54283500,315.960836,Overstock


In [52]:
inventory_analysis["excess_stock_units"] = np.maximum(
    inventory_analysis["current_stock"] - inventory_analysis["max_stock"],
    0
)

inventory_analysis["stock_above_reorder"] = (
    inventory_analysis["current_stock"]
    - inventory_analysis["reorder_level"]
)

inventory_analysis["inventory_value"] = (
    inventory_analysis["current_stock"]
    * inventory_analysis["unit_cost"]
)

inventory_analysis["excess_inventory_value"] = (
    inventory_analysis["excess_stock_units"]
    * inventory_analysis["unit_cost"]
)

inventory_analysis["stock_status"] = np.select(
    [
        inventory_analysis["current_stock"]
        <= inventory_analysis["safety_stock"],

        inventory_analysis["current_stock"]
        <= inventory_analysis["reorder_level"],

        inventory_analysis["current_stock"]
        <= inventory_analysis["max_stock"]
    ],
    [
        "Critical",
        "Reorder Required",
        "Healthy"
    ],
    default="Overstock"
)

inventory_analysis["stock_to_max_ratio"] = (
    inventory_analysis["current_stock"]
    / inventory_analysis["max_stock"]
)

In [53]:
print(
    "Total current inventory value:",
    round(inventory_analysis["inventory_value"].sum(), 2)
)

print(
    "Total excess inventory value:",
    round(inventory_analysis["excess_inventory_value"].sum(), 2)
)

print(
    "Average stock-to-max ratio:",
    round(inventory_analysis["stock_to_max_ratio"].mean(), 2)
)

Total current inventory value: 281292677410
Total excess inventory value: 280460601920
Average stock-to-max ratio: 385.04


In [54]:
inventory_analysis["stock_to_max_ratio"].describe()

count    180.000000
mean     385.039933
std      155.738138
min      177.408127
25%      259.168559
50%      349.372795
75%      477.508863
max      854.233083
Name: stock_to_max_ratio, dtype: float64

In [55]:
inventory_analysis[
    [
        "product_id",
        "product_name",
        "branch_id",
        "current_stock",
        "max_stock",
        "stock_to_max_ratio",
        "excess_inventory_value"
    ]
].sort_values(
    "stock_to_max_ratio",
    ascending=False
).head(10)

,product_id,product_name,branch_id,current_stock,max_stock,stock_to_max_ratio,excess_inventory_value
147,P025,Cabin Glass CG-20,HYD001,113613,133,854.233083,635488000
89,P015,Engine Block EB-600,AHM001,105135,131,802.557252,10027882000
135,P023,Idler Wheel IW-55,HYD001,105798,134,789.537313,972108800
61,P011,Hydraulic Cylinder HC-200,PUN001,108691,138,787.615942,2333889500
73,P013,Seal Kit SK-15,PUN001,104221,133,783.616541,98883600
114,P020,Pivot Pin PP-12,DEL001,104360,140,745.428571,78165000
158,P027,O-Ring Set OR-50,CHN001,97973,133,736.639098,25438400
115,P020,Pivot Pin PP-12,PUN001,102734,140,733.814286,76945500
79,P014,Transmission Assembly TA-800,PUN001,113386,164,691.378049,7755707000
71,P012,Brake Pad BP-40,AHM001,106861,156,685.006410,192069000


In [56]:
excess_value_percent = (
    inventory_analysis["excess_inventory_value"].sum()
    / inventory_analysis["inventory_value"].sum()
    * 100
)

print(
    "Excess inventory value percentage:",
    round(excess_value_percent, 2),
    "%"
)

Excess inventory value percentage: 99.7 %


### Initial Inventory-Level Finding

All 180 product-branch combinations are currently above their configured maximum stock levels.

The calculated current inventory value is approximately 281.29 billion, while approximately 280.46 billion is classified as inventory above configured maximum levels.

The average current-stock-to-maximum-stock ratio is approximately 385 times the configured maximum.

Although these figures are internally consistent with the latest stock-ledger balances, their scale is unusually large. Therefore, this finding should not yet be interpreted solely as operational overstocking.

Further investigation is required to determine whether the simulated dataset contains a structural difference between cumulative stock-ledger balances and the inventory-policy threshold fields.

In [57]:
inventory_analysis["stock_to_max_ratio"].describe()

count    180.000000
mean     385.039933
std      155.738138
min      177.408127
25%      259.168559
50%      349.372795
75%      477.508863
max      854.233083
Name: stock_to_max_ratio, dtype: float64

### Inventory Threshold Scale Limitation

Further analysis of the `stock_to_max_ratio` confirmed that the unusually high values are systematic rather than being caused by a small number of outliers.

Summary statistics:

- Minimum ratio: approximately 177×
- Median ratio: approximately 349×
- Mean ratio: approximately 385×
- Maximum ratio: approximately 854×

Because every product-branch combination is far above its configured maximum stock level, the `current_stock` and stock-threshold fields may operate on different scales or represent different business concepts within the simulated dataset.

Therefore, `stock_to_max_ratio`, excess stock, and excess inventory value will be treated as exploratory indicators rather than final business KPIs until the relationship between these fields is better understood.

In [58]:
warehouse_analysis = branches.copy()

In [59]:
branch_inventory = (
    inventory_analysis
    .groupby("branch_id")
    .agg(
        total_current_stock=("current_stock", "sum"),
        total_inventory_value=("inventory_value", "sum"),
        total_excess_inventory_value=("excess_inventory_value", "sum"),
        product_count=("product_id", "nunique")
    )
    .reset_index()
)

In [60]:
branch_sales = (
    sales_analysis
    .groupby("branch_id")
    .agg(
        total_sales_value=("line_total", "sum"),
        total_units_sold=("quantity", "sum"),
        total_gross_profit=("gross_profit", "sum"),
        sales_order_count=("so_id", "nunique"),
        customer_count=("customer_id", "nunique")
    )
    .reset_index()
)

In [61]:
branch_stock_movements = (
    stock_ledger
    .groupby(["branch_id", "movement_type"])["quantity"]
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

In [62]:
branch_stock_movements = branch_stock_movements.rename(
    columns={
        "IN": "stock_in_units",
        "OUT": "stock_out_units",
        "ADJUSTMENT": "stock_adjustment_units"
    }
)

In [63]:
warehouse_analysis = warehouse_analysis.merge(
    branch_inventory,
    on="branch_id",
    how="left",
    validate="one_to_one"
)

warehouse_analysis = warehouse_analysis.merge(
    branch_sales,
    on="branch_id",
    how="left",
    validate="one_to_one"
)

warehouse_analysis = warehouse_analysis.merge(
    branch_stock_movements,
    on="branch_id",
    how="left",
    validate="one_to_one"
)

In [64]:
print("Branches:", len(branches))
print("Warehouse analysis rows:", len(warehouse_analysis))
print("Difference:", len(warehouse_analysis) - len(branches))

warehouse_analysis.shape

Branches: 6
Warehouse analysis rows: 6
Difference: 0


(6, 26)

In [65]:
warehouse_analysis["sales_per_employee"] = (
    warehouse_analysis["total_sales_value"]
    / warehouse_analysis["total_employees"]
)

warehouse_analysis["operating_cost_ratio"] = (
    warehouse_analysis["monthly_operational_cost"]
    / warehouse_analysis["avg_monthly_revenue"]
)

warehouse_analysis["stock_out_to_in_ratio"] = (
    warehouse_analysis["stock_out_units"]
    / warehouse_analysis["stock_in_units"]
)

In [66]:
warehouse_analysis[
    [
        "branch_id",
        "branch_name",
        "total_sales_value",
        "total_units_sold",
        "total_inventory_value",
        "stock_in_units",
        "stock_out_units",
        "sales_per_employee",
        "operating_cost_ratio",
        "stock_out_to_in_ratio"
    ]
]

,branch_id,branch_name,total_sales_value,total_units_sold,total_inventory_value,stock_in_units,stock_out_units,sales_per_employee,operating_cost_ratio,stock_out_to_in_ratio
0,DEL001,Delhi Central,3865954470,204734,47709094440,3453132,169332,4.202124e+07,0.609182,0.049037
1,PUN001,Pune Distribution,3741098180,201749,47728085910,3415584,165312,5.055538e+07,0.604677,0.048399
2,CHN001,Chennai South Hub,5192125100,280446,47798163500,3461260,231544,6.410031e+07,0.599635,0.066896
3,HYD001,Hyderabad Logistics,3452672510,189163,47224691630,3394339,153895,6.057320e+07,0.608228,0.045339
4,KOL001,Kolkata East Depot,4787656160,256095,44702846930,3328556,210007,9.770727e+07,0.602120,0.063093
5,AHM001,Ahmedabad West Hub,4407886140,236347,46129795000,3344924,194742,6.996645e+07,0.621709,0.058220


### Initial Warehouse-Level Findings

The branch-level analytical dataset combines inventory, sales, workforce, operating-cost, and stock-movement information for all six branches.

Initial observations include:

- Chennai South Hub currently has the highest total sales value.
- Hyderabad Logistics has the lowest total sales value among the six branches.
- Kolkata East Depot has the highest sales value per employee, suggesting relatively strong workforce productivity based on this metric.
- Operating-cost ratios are relatively similar across branches, ranging from approximately 0.60 to 0.62.
- Ahmedabad West Hub has the highest operating-cost ratio, while Chennai South Hub has the lowest.
- Stock-out-to-stock-in ratios are low across all branches.

The low stock-out-to-stock-in ratios should be interpreted cautiously because the stock ledger shows substantially higher cumulative incoming quantities than outgoing quantities. This pattern is consistent with the unusually high current inventory levels identified earlier.

These metrics will be used as exploratory indicators rather than final warehouse-performance scores until additional operational context is considered.

In [67]:
warehouse_ranking = warehouse_analysis[
    [
        "branch_id",
        "branch_name",
        "total_sales_value",
        "total_units_sold",
        "sales_per_employee",
        "operating_cost_ratio",
        "stock_out_to_in_ratio"
    ]
].copy()

warehouse_ranking["sales_rank"] = (
    warehouse_ranking["total_sales_value"]
    .rank(ascending=False, method="dense")
    .astype(int)
)

warehouse_ranking["productivity_rank"] = (
    warehouse_ranking["sales_per_employee"]
    .rank(ascending=False, method="dense")
    .astype(int)
)

warehouse_ranking["cost_efficiency_rank"] = (
    warehouse_ranking["operating_cost_ratio"]
    .rank(ascending=True, method="dense")
    .astype(int)
)

warehouse_ranking.sort_values("sales_rank")

,branch_id,branch_name,total_sales_value,total_units_sold,sales_per_employee,operating_cost_ratio,stock_out_to_in_ratio,sales_rank,productivity_rank,cost_efficiency_rank
2,CHN001,Chennai South Hub,5192125100,280446,6.410031e+07,0.599635,0.066896,1,3,1
4,KOL001,Kolkata East Depot,4787656160,256095,9.770727e+07,0.602120,0.063093,2,1,2
5,AHM001,Ahmedabad West Hub,4407886140,236347,6.996645e+07,0.621709,0.058220,3,2,6
0,DEL001,Delhi Central,3865954470,204734,4.202124e+07,0.609182,0.049037,4,6,5
1,PUN001,Pune Distribution,3741098180,201749,5.055538e+07,0.604677,0.048399,5,5,3
3,HYD001,Hyderabad Logistics,3452672510,189163,6.057320e+07,0.608228,0.045339,6,4,4


In [68]:
supplier_po_summary = (
    po_header
    .groupby("supplier_id")
    .agg(
        total_po_count=("po_id", "nunique"),
        total_procurement_value=("grand_total", "sum"),
        avg_po_value=("grand_total", "mean"),
        avg_delivery_delay_days=("delivery_delay_days", "mean")
    )
    .reset_index()
)

In [69]:
supplier_delivery = (
    po_header
    .groupby(["supplier_id", "delivery_performance"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

In [70]:
supplier_analysis = suppliers.merge(
    supplier_po_summary,
    on="supplier_id",
    how="left",
    validate="one_to_one"
)

supplier_analysis = supplier_analysis.merge(
    supplier_delivery,
    on="supplier_id",
    how="left",
    validate="one_to_one"
)

In [71]:
supplier_analysis["on_time_rate"] = (
    supplier_analysis["On Time"]
    / supplier_analysis["total_po_count"]
    * 100
)

supplier_analysis["late_rate"] = (
    supplier_analysis["Late"]
    / supplier_analysis["total_po_count"]
    * 100
)

supplier_analysis["cancel_rate"] = (
    supplier_analysis["Cancelled"]
    / supplier_analysis["total_po_count"]
    * 100
)

In [72]:
print("Suppliers:", len(suppliers))
print("Supplier analysis rows:", len(supplier_analysis))
print("Difference:", len(supplier_analysis) - len(suppliers))

supplier_analysis.shape

Suppliers: 8
Supplier analysis rows: 8
Difference: 0


(8, 23)

In [73]:
supplier_analysis[
    [
        "supplier_id",
        "supplier_name",
        "supplier_type",
        "reliability_score",
        "total_po_count",
        "total_procurement_value",
        "on_time_rate",
        "late_rate",
        "cancel_rate",
        "avg_delivery_delay_days"
    ]
].sort_values(
    "on_time_rate",
    ascending=False
)

,supplier_id,supplier_name,supplier_type,reliability_score,total_po_count,total_procurement_value,on_time_rate,late_rate,cancel_rate,avg_delivery_delay_days
4,SUP0005,Tianjin OEM Supplies,OEM,3,3044,5.702122e+10,73.357424,13.272011,9.526938,0.485476
5,SUP0006,Qingdao Distributor Supplies,Distributor,3,3038,5.509357e+10,72.942725,13.396972,9.414088,0.501090
7,SUP0008,Hangzhou OEM Supplies,OEM,4,3010,5.535490e+10,72.458472,14.285714,9.568106,0.545555
6,SUP0007,Suzhou Local Vendor Supplies,Local Vendor,5,3028,5.559772e+10,72.457067,13.804491,9.940555,0.528053
1,SUP0002,Guangzhou Local Vendor Supplies,Local Vendor,5,2937,5.327595e+10,72.420838,13.483146,10.282601,0.516129
3,SUP0004,Ningbo Distributor Supplies,Distributor,3,2966,5.406021e+10,72.117330,13.621038,10.182063,0.497372
0,SUP0001,Shenzhen OEM Supplies,OEM,5,2944,5.564297e+10,72.112772,13.586957,9.884511,0.496419
2,SUP0003,Shanghai Distributor Supplies,Distributor,4,3033,5.563977e+10,71.843060,13.452028,10.220903,0.531399


In [75]:
supplier_analysis["received_po_count"] = (
    supplier_analysis["On Time"]
    + supplier_analysis["Late"]
    + supplier_analysis["Early"]
)

In [76]:
supplier_analysis["received_on_time_rate"] = (
    supplier_analysis["On Time"]
    / supplier_analysis["received_po_count"]
    * 100
)

supplier_analysis["received_late_rate"] = (
    supplier_analysis["Late"]
    / supplier_analysis["received_po_count"]
    * 100
)

supplier_analysis["received_early_rate"] = (
    supplier_analysis["Early"]
    / supplier_analysis["received_po_count"]
    * 100
)

### Initial Supplier Findings

Supplier performance is relatively similar across the eight suppliers.

Tianjin OEM Supplies currently has the highest overall on-time delivery rate, while Shanghai Distributor Supplies has the lowest.

Hangzhou OEM Supplies shows the highest late-delivery rate, and Guangzhou Local Vendor Supplies has the highest cancellation rate.

Average delivery delay among received orders is relatively small across suppliers.

Two delivery-rate perspectives are retained:

- Overall on-time rate uses all purchase orders, including cancellations.
- Received on-time rate evaluates delivery reliability only among purchase orders that were actually received.

This distinction prevents cancellations from being incorrectly interpreted as late or on-time deliveries.

In [77]:
supplier_analysis[
    [
        "supplier_id",
        "supplier_name",
        "reliability_score",
        "total_po_count",
        "received_po_count",
        "on_time_rate",
        "received_on_time_rate",
        "received_late_rate",
        "cancel_rate",
        "avg_delivery_delay_days"
    ]
].sort_values(
    "received_on_time_rate",
    ascending=False
)

,supplier_id,supplier_name,reliability_score,total_po_count,received_po_count,on_time_rate,received_on_time_rate,received_late_rate,cancel_rate,avg_delivery_delay_days
4,SUP0005,Tianjin OEM Supplies,3,3044,2754,73.357424,81.082062,14.669572,9.526938,0.485476
1,SUP0002,Guangzhou Local Vendor Supplies,5,2937,2635,72.420838,80.721063,15.028463,10.282601,0.516129
5,SUP0006,Qingdao Distributor Supplies,3,3038,2752,72.942725,80.523256,14.789244,9.414088,0.501090
6,SUP0007,Suzhou Local Vendor Supplies,5,3028,2727,72.457067,80.454712,15.328199,9.940555,0.528053
3,SUP0004,Ningbo Distributor Supplies,3,2966,2664,72.117330,80.292793,15.165165,10.182063,0.497372
7,SUP0008,Hangzhou OEM Supplies,4,3010,2722,72.458472,80.124908,15.797208,9.568106,0.545555
0,SUP0001,Shenzhen OEM Supplies,5,2944,2653,72.112772,80.022616,15.077271,9.884511,0.496419
2,SUP0003,Shanghai Distributor Supplies,4,3033,2723,71.843060,80.022035,14.983474,10.220903,0.531399


### Refined Supplier Delivery Reliability

Supplier delivery performance was evaluated using only received purchase orders in addition to the overall purchase-order rate.

The received on-time delivery rates are relatively similar across all suppliers, at approximately 80%–81%.

Tianjin OEM Supplies has the highest received on-time rate, while Shanghai Distributor Supplies has the lowest.

Hangzhou OEM Supplies shows the highest received late-delivery rate.

Because the differences between suppliers are relatively small, supplier performance should later be assessed using multiple indicators, including delivery reliability, cancellation rate, procurement value, reliability score, and lead time, rather than relying on a single metric.

In [78]:
customer_analysis = customers.copy()

In [79]:
delivered_sales = sales_analysis[
    sales_analysis["order_status"] == "Delivered"
].copy()

In [80]:
delivered_sales = sales_analysis[
    sales_analysis["order_status"] == "Delivered"
].copy()

In [81]:
customer_sales_summary = (
    delivered_sales
    .groupby("customer_id")
    .agg(
        total_sales_value=("line_total", "sum"),
        total_units_purchased=("quantity", "sum"),
        total_orders=("so_id", "nunique"),
        unique_products_purchased=("product_id", "nunique"),
        first_order_date=("order_date", "min"),
        last_order_date=("order_date", "max")
    )
    .reset_index()
)

In [82]:
customer_order_values = (
    delivered_sales
    .groupby(["customer_id", "so_id"])
    .agg(
        order_value=("line_total", "sum")
    )
    .reset_index()
)

In [83]:
customer_aov = (
    customer_order_values
    .groupby("customer_id")
    .agg(
        average_order_value=("order_value", "mean")
    )
    .reset_index()
)

In [84]:
customer_analysis = customer_analysis.merge(
    customer_sales_summary,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

customer_analysis = customer_analysis.merge(
    customer_aov,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

In [85]:
print("Customers:", len(customers))
print("Customer analysis rows:", len(customer_analysis))
print("Difference:", len(customer_analysis) - len(customers))

customer_analysis.shape

Customers: 500
Customer analysis rows: 500
Difference: 0


(500, 22)

In [86]:
customer_analysis["purchase_frequency"] = (
    customer_analysis["total_orders"]
)

In [87]:
customer_analysis["customer_tenure_days"] = (
    customer_analysis["last_order_date"]
    - customer_analysis["customer_since"]
).dt.days

In [88]:
analysis_date = delivered_sales["order_date"].max()

customer_analysis["recency_days"] = (
    analysis_date
    - customer_analysis["last_order_date"]
).dt.days

In [89]:
customer_analysis["avg_units_per_order"] = (
    customer_analysis["total_units_purchased"]
    / customer_analysis["total_orders"]
)

In [90]:
customer_analysis[
    [
        "customer_id",
        "customer_type",
        "industry_segment",
        "total_sales_value",
        "total_orders",
        "average_order_value",
        "total_units_purchased",
        "unique_products_purchased",
        "recency_days",
        "customer_tenure_days",
        "avg_units_per_order"
    ]
].head(10)

,customer_id,customer_type,industry_segment,total_sales_value,total_orders,average_order_value,total_units_purchased,unique_products_purchased,recency_days,customer_tenure_days,avg_units_per_order
0,C0001,Retail,Construction,43201680,34,1.270638e+06,2123,30,46,219,62.441176
1,C0002,Fleet Owner,Manufacturing,56261100,51,1.103159e+06,3253,30,115,962,63.784314
2,C0003,Government,Logistics,35112260,38,9.240068e+05,2164,30,62,2815,56.947368
3,C0004,Fleet Owner,Logistics,47754210,46,1.038135e+06,2879,30,0,2568,62.586957
4,C0005,Dealer,Construction,58882280,45,1.308495e+06,3231,30,88,435,71.800000
5,C0006,Government,Construction,51412280,40,1.285307e+06,2737,30,120,1054,68.425000
6,C0007,Corporate,Manufacturing,28648240,31,9.241368e+05,1908,30,58,2058,61.548387
7,C0008,Corporate,Logistics,37972100,34,1.116826e+06,2280,30,168,3310,67.058824
8,C0009,Corporate,Rental,54041610,34,1.589459e+06,2677,30,57,942,78.735294
9,C0010,Dealer,Construction,49509380,46,1.076291e+06,2985,30,34,3255,64.891304


In [91]:
print(
    "Customers with no delivered orders:",
    customer_analysis["total_orders"].isna().sum()
)

print(
    "Average customer sales value:",
    round(customer_analysis["total_sales_value"].mean(), 2)
)

print(
    "Average number of delivered orders:",
    round(customer_analysis["total_orders"].mean(), 2)
)

print(
    "Average order value:",
    round(customer_analysis["average_order_value"].mean(), 2)
)

Customers with no delivered orders: 0
Average customer sales value: 45917974.86
Average number of delivered orders: 36.07
Average order value: 1273690.24


### Initial Customer-Level Findings

The customer-level analytical dataset contains all 500 customers.

All customers have at least one delivered sales order, indicating that every customer in the customer master has recorded completed purchasing activity.

Initial customer behavior metrics show:

- Average customer sales value is approximately 45.92 million.
- Customers place approximately 36 delivered orders on average.
- Average order value is approximately 1.27 million.
- No customers currently have zero delivered orders.

These averages provide a high-level view of customer activity. More detailed customer analytics, including RFM segmentation, high-value customer identification, purchase concentration, and at-risk customer analysis, will be performed in a later project phase.

In [92]:
import os

os.makedirs("analysis_ready_data", exist_ok=True)

In [93]:
sales_analysis.to_csv(
    "analysis_ready_data/sales_analysis.csv",
    index=False
)

procurement_analysis.to_csv(
    "analysis_ready_data/procurement_analysis.csv",
    index=False
)

inventory_analysis.to_csv(
    "analysis_ready_data/inventory_analysis.csv",
    index=False
)

warehouse_analysis.to_csv(
    "analysis_ready_data/warehouse_analysis.csv",
    index=False
)

supplier_analysis.to_csv(
    "analysis_ready_data/supplier_analysis.csv",
    index=False
)

customer_analysis.to_csv(
    "analysis_ready_data/customer_analysis.csv",
    index=False
)

print("All Week 3 analytical datasets exported successfully.")

All Week 3 analytical datasets exported successfully.


In [94]:
for file in os.listdir("analysis_ready_data"):
    print(file)

customer_analysis.csv
inventory_analysis.csv
procurement_analysis.csv
sales_analysis.csv
supplier_analysis.csv
warehouse_analysis.csv
